In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🏢 Enterprise Integration Patterns Guide

**Building scalable enterprise systems with integration patterns, adapters, transformations, and legacy system connectivity**

This guide covers:
- Enterprise Integration Patterns (EIP)
- Enterprise Service Bus (ESB)
- Adapter pattern for legacy systems
- Message transformations
- Event-driven integration
- Service orchestration
- API management for enterprises
- Data consistency in distributed systems
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Enterprise Integration Patterns

### Message-Based Integration
```python
from dataclasses import dataclass
from enum import Enum
from typing import List, Dict, Callable
from datetime import datetime
import uuid

class MessageType(Enum):
    COMMAND = "command"
    EVENT = "event"
    QUERY = "query"
    RESPONSE = "response"

@dataclass
class Message:
    """Standard enterprise message"""
    message_id: str
    message_type: MessageType
    source_service: str
    destination_service: str
    payload: Dict
    timestamp: str
    correlation_id: str = None
    reply_to: str = None
    
    def __post_init__(self):
        if self.message_id is None:
            self.message_id = str(uuid.uuid4())
        if self.correlation_id is None:
            self.correlation_id = self.message_id
        if self.timestamp is None:
            self.timestamp = datetime.now().isoformat()

class MessageRouter:
    """Route messages to appropriate services"""
    
    def __init__(self):
        self.routes = {}
        self.handlers = {}
    
    def register_route(self, message_type: str, service_name: str, handler: Callable):
        """Register message route"""
        key = f"{message_type}:{service_name}"
        self.routes[key] = True
        self.handlers[key] = handler
    
    def route_message(self, message: Message) -> any:
        """Route message to handler"""
        key = f"{message.message_type.value}:{message.destination_service}"
        
        if key not in self.handlers:
            raise ValueError(f"No handler for {key}")
        
        handler = self.handlers[key]
        return handler(message)
    
    def publish_event(self, event_name: str, data: Dict):
        """Publish event for all interested subscribers"""
        message = Message(
            message_id=None,
            message_type=MessageType.EVENT,
            source_service="system",
            destination_service="*",
            payload={'event': event_name, 'data': data}
        )
        
        # Notify all interested parties
        for key in self.handlers:
            if event_name in key:
                self.route_message(message)

class MessageBroker:
    """Central message broker for enterprise"""
    
    def __init__(self):
        self.router = MessageRouter()
        self.message_queue = []
        self.dead_letter_queue = []
    
    def send_message(self, message: Message) -> bool:
        """Send message through broker"""
        try:
            result = self.router.route_message(message)
            self.message_queue.append({
                'message': message,
                'result': result,
                'status': 'delivered'
            })
            return True
        except Exception as e:
            # Send to dead letter queue
            self.dead_letter_queue.append({
                'message': message,
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            })
            return False
    
    def retry_dead_letters(self):
        """Retry failed messages"""
        retried = 0
        
        for dead_letter in self.dead_letter_queue[:]:
            try:
                self.send_message(dead_letter['message'])
                self.dead_letter_queue.remove(dead_letter)
                retried += 1
            except:
                pass
        
        return retried
```

### Publish-Subscribe Pattern
```python
class PublisherSubscriber:
    """Publish-Subscribe integration pattern"""
    
    def __init__(self):
        self.subscribers = {}
    
    def subscribe(self, topic: str, callback: Callable):
        """Subscribe to topic"""
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        
        self.subscribers[topic].append(callback)
    
    def unsubscribe(self, topic: str, callback: Callable):
        """Unsubscribe from topic"""
        if topic in self.subscribers:
            self.subscribers[topic].remove(callback)
    
    def publish(self, topic: str, message: Dict):
        """Publish message to all subscribers"""
        if topic not in self.subscribers:
            print(f"No subscribers for topic: {topic}")
            return
        
        for callback in self.subscribers[topic]:
            try:
                callback(message)
            except Exception as e:
                print(f"Error calling subscriber: {str(e)}")

# Usage
pubsub = PublisherSubscriber()

def on_order_created(message):
    print(f"Order created: {message['order_id']}")

pubsub.subscribe("orders:created", on_order_created)
pubsub.publish("orders:created", {"order_id": "123", "amount": 100})
```

### Request-Response Pattern
```python
class RequestResponseIntegration:
    """Request-Response integration pattern"""
    
    def __init__(self):
        self.pending_requests = {}
    
    def send_request(self, service_name: str, request_data: Dict, timeout: int = 5) -> Dict:
        """Send request and wait for response"""
        request_id = str(uuid.uuid4())
        
        request = Message(
            message_id=request_id,
            message_type=MessageType.QUERY,
            source_service="system",
            destination_service=service_name,
            payload=request_data,
            reply_to="system"
        )
        
        self.pending_requests[request_id] = {'pending': True}
        
        # Send request (simulated)
        response = self._simulate_service_call(service_name, request)
        
        self.pending_requests[request_id] = {'response': response, 'pending': False}
        
        return response
    
    def _simulate_service_call(self, service_name: str, request: Message) -> Dict:
        """Simulate service processing"""
        # In real system, would be RPC call
        return {
            'request_id': request.message_id,
            'service': service_name,
            'result': 'success',
            'data': {}
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Enterprise Service Bus (ESB)

### Simple ESB Implementation
```python
class EnterpriseServiceBus:
    """Enterprise Service Bus for integration"""
    
    def __init__(self):
        self.services = {}
        self.routes = {}
        self.transformers = {}
        self.message_history = []
    
    def register_service(self, service_name: str, service_handler: Callable):
        """Register service on ESB"""
        self.services[service_name] = service_handler
        print(f"Registered service: {service_name}")
    
    def register_route(self, source: str, destination: str, transformer: Callable = None):
        """Register message route"""
        key = f"{source}->{destination}"
        self.routes[key] = destination
        
        if transformer:
            self.transformers[key] = transformer
    
    def send_message(self, source: str, destination: str, message: Dict) -> Dict:
        """Send message through ESB"""
        
        # Log message
        self.message_history.append({
            'from': source,
            'to': destination,
            'message': message,
            'timestamp': datetime.now().isoformat()
        })
        
        # Get route
        key = f"{source}->{destination}"
        if key not in self.routes:
            return {'error': f'No route from {source} to {destination}'}
        
        # Transform if needed
        transformed = message
        if key in self.transformers:
            transformed = self.transformers[key](message)
        
        # Route to destination
        if destination in self.services:
            result = self.services[destination](transformed)
            return result
        
        return {'error': f'Destination {destination} not found'}
    
    def get_message_history(self, service_name: str = None) -> List[Dict]:
        """Get message history"""
        if service_name:
            return [m for m in self.message_history if m['from'] == service_name or m['to'] == service_name]
        return self.message_history

class MessageTransformer:
    """Transform messages between services"""
    
    @staticmethod
    def json_to_xml(json_data: Dict) -> str:
        """Convert JSON to XML"""
        import xml.etree.ElementTree as ET
        
        root = ET.Element('root')
        for key, value in json_data.items():
            elem = ET.SubElement(root, key)
            elem.text = str(value)
        
        return ET.tostring(root, encoding='unicode')
    
    @staticmethod
    def xml_to_json(xml_data: str) -> Dict:
        """Convert XML to JSON"""
        import xml.etree.ElementTree as ET
        
        root = ET.fromstring(xml_data)
        result = {}
        
        for elem in root:
            result[elem.tag] = elem.text
        
        return result
    
    @staticmethod
    def normalize_date_format(data: Dict, old_format: str, new_format: str) -> Dict:
        """Normalize date formats between systems"""
        from datetime import datetime
        
        result = data.copy()
        
        for key in ['date', 'timestamp', 'created_at', 'updated_at']:
            if key in result:
                try:
                    dt = datetime.strptime(result[key], old_format)
                    result[key] = dt.strftime(new_format)
                except:
                    pass
        
        return result
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Adapter Pattern for Legacy Systems

### Legacy System Adapter
```python
class LegacySystemAdapter:
    """Adapter for legacy system integration"""
    
    def __init__(self, legacy_system_name: str):
        self.legacy_system = legacy_system_name
        self.mapping = {}
    
    def map_field(self, external_field: str, legacy_field: str):
        """Map external field to legacy field"""
        self.mapping[external_field] = legacy_field
    
    def translate_request_to_legacy(self, external_request: Dict) -> Dict:
        """Translate modern request to legacy format"""
        legacy_request = {}
        
        for external_field, value in external_request.items():
            if external_field in self.mapping:
                legacy_field = self.mapping[external_field]
                legacy_request[legacy_field] = value
        
        return legacy_request
    
    def translate_response_from_legacy(self, legacy_response: Dict) -> Dict:
        """Translate legacy response to modern format"""
        external_response = {}
        
        # Reverse mapping
        reverse_mapping = {v: k for k, v in self.mapping.items()}
        
        for legacy_field, value in legacy_response.items():
            if legacy_field in reverse_mapping:
                external_field = reverse_mapping[legacy_field]
                external_response[external_field] = value
        
        return external_response
    
    def call_legacy_system(self, operation: str, params: Dict) -> Dict:
        """Call legacy system through adapter"""
        
        # Translate request
        legacy_params = self.translate_request_to_legacy(params)
        
        print(f"Calling legacy system {self.legacy_system} with {operation}")
        print(f"  Original params: {params}")
        print(f"  Legacy params: {legacy_params}")
        
        # Simulate legacy system call
        legacy_response = {
            'legacy_id': legacy_params.get('id'),
            'legacy_status': 'success',
            'legacy_data': legacy_params
        }
        
        # Translate response
        response = self.translate_response_from_legacy(legacy_response)
        
        return response

# Usage
adapter = LegacySystemAdapter("OldERPSystem")
adapter.map_field("customer_id", "cust_no")
adapter.map_field("order_date", "ord_dt")

response = adapter.call_legacy_system("create_order", {
    "customer_id": "CUST123",
    "order_date": "2024-01-15"
})
```

### Data Migration Adapter
```python
class DataMigrationAdapter:
    """Migrate data from legacy system"""
    
    def __init__(self, source_system: str, target_system: str):
        self.source = source_system
        self.target = target_system
        self.migration_mapping = {}
    
    def define_migration_mapping(self, source_table: str, target_table: str, field_mapping: Dict):
        """Define how to migrate data"""
        self.migration_mapping[source_table] = {
            'target_table': target_table,
            'field_mapping': field_mapping
        }
    
    def migrate_data(self, source_table: str, data: List[Dict]) -> List[Dict]:
        """Migrate batch of records"""
        
        if source_table not in self.migration_mapping:
            raise ValueError(f"No mapping for {source_table}")
        
        mapping_info = self.migration_mapping[source_table]
        target_table = mapping_info['target_table']
        field_mapping = mapping_info['field_mapping']
        
        migrated = []
        
        for record in data:
            migrated_record = {}
            
            for source_field, target_field in field_mapping.items():
                if source_field in record:
                    value = record[source_field]
                    
                    # Apply transformation if needed
                    if callable(target_field):
                        value = target_field(value)
                    else:
                        migrated_record[target_field] = value
            
            migrated.append(migrated_record)
        
        return migrated
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Event-Driven Integration

### Event Store Pattern
```python
class EventStore:
    """Store events for event sourcing"""
    
    def __init__(self):
        self.events = []
        self.snapshots = {}
    
    def append_event(self, aggregate_id: str, event_type: str, data: Dict):
        """Append event to store"""
        event = {
            'aggregate_id': aggregate_id,
            'event_type': event_type,
            'data': data,
            'timestamp': datetime.now().isoformat(),
            'version': len(self.events) + 1
        }
        
        self.events.append(event)
        return event
    
    def get_events(self, aggregate_id: str, from_version: int = 0) -> List[Dict]:
        """Get events for aggregate"""
        return [e for e in self.events 
                if e['aggregate_id'] == aggregate_id and e['version'] > from_version]
    
    def create_snapshot(self, aggregate_id: str, version: int, state: Dict):
        """Create snapshot for faster replay"""
        self.snapshots[aggregate_id] = {
            'version': version,
            'state': state,
            'timestamp': datetime.now().isoformat()
        }
    
    def rebuild_state(self, aggregate_id: str) -> Dict:
        """Rebuild aggregate state from events"""
        state = {}
        
        # Check for recent snapshot
        if aggregate_id in self.snapshots:
            snapshot = self.snapshots[aggregate_id]
            state = snapshot['state'].copy()
            from_version = snapshot['version']
        else:
            from_version = 0
        
        # Replay events since snapshot
        events = self.get_events(aggregate_id, from_version)
        
        for event in events:
            state = self._apply_event(state, event)
        
        return state
    
    def _apply_event(self, state: Dict, event: Dict) -> Dict:
        """Apply event to state"""
        event_type = event['event_type']
        data = event['data']
        
        if event_type == 'OrderCreated':
            return {'order_id': data['order_id'], 'status': 'created'}
        elif event_type == 'OrderConfirmed':
            state['status'] = 'confirmed'
            return state
        
        return state
```
</VSCode.Cell>

<MySCode.Cell language="markdown">
## 5. Enterprise Integration Checklist

✅ **Messaging**
- [ ] Message format standardized
- [ ] Message routing configured
- [ ] Error handling implemented
- [ ] Dead letter queue monitored
- [ ] Message auditing enabled

✅ **ESB**
- [ ] ESB deployed and configured
- [ ] Services registered
- [ ] Routes defined
- [ ] Transformations tested
- [ ] Performance monitored

✅ **Legacy Integration**
- [ ] Legacy systems mapped
- [ ] Adapters built and tested
- [ ] Data consistency maintained
- [ ] Migration plan documented
- [ ] Rollback procedures ready

✅ **Event Integration**
- [ ] Event store set up
- [ ] Event publishers identified
- [ ] Event subscribers configured
- [ ] Event schema versioned
- [ ] Replay capabilities tested

✅ **Operations**
- [ ] Monitoring dashboard created
- [ ] Alerts configured
- [ ] SLAs defined
- [ ] Runbooks documented
- [ ] Incident response procedures ready
</MySCode.Cell>
```